#Reading Data From Bronze

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.types import DateType

In [0]:
df_erp_cust = spark.table('workspace.bronze.erp_cust_az12')

In [0]:
display(df_erp_cust)

In [0]:
for field in df_erp_cust.schema.fields:
  print(field.name, field.dataType)

#Transforming Data

Transforming sales table
- trim string values - good practice
- normalization for status & gender
- column names are not friendly

##Trimming

In [0]:
#Triming the string column 

for field in df_erp_cust.schema.fields:
    if field.dataType == StringType():
        df_erp_cust = df_erp_cust.withColumn(field.name, trim(col(field.name)))

## Correcting the gender column

In [0]:
#standardizing the columns 
#lets get all the unique values to understand what all Values we have in gender column 
df_erp_cust.select("GEN").distinct().show()

In [0]:
#standardizing the columns
df_erp_cust = (
    df_erp_cust.withColumn(
        "GEN",
        when(upper(col("GEN")).isin('M', 'MALE'), 'Male')
        .when(upper(col("GEN")).isin('F', 'FEMALE'), 'Female')
        .otherwise("n/a")
    )
)

In [0]:
#validating
df_erp_cust.select("GEN").distinct().show()

In [0]:
display(df_erp_cust)

In [0]:
df_erp_cust.select(count(col("BDATE").isNull()).alias("birth_date")).show()

In [0]:
#getting the column name
df_erp_cust.columns

In [0]:
#renaming the column names
remap_name = {
    "CID": "customer_key",
    "BDATE": "birth_date",
    "GEN": "gender",
}
for old_val, new_val in remap_name.items():
    df_erp_cust = df_erp_cust.withColumnRenamed(old_val, new_val)
#validating
df_erp_cust.columns

#Writing to Silver 

In [0]:
df_erp_cust.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_cust")

In [0]:
%sql
select * from workspace.silver.erp_cust Limit 10